In [1]:
%run ./nb_utils_api_acto_gestao

StatementMeta(, 7fe91b3e-73cc-45a2-bcc4-03436fb0c024, 6, Finished, Available, Finished, True)

In [2]:
import numpy as np
import json
from pathlib import Path
pd.set_option("display.max_columns", None)

ARQUIVO_AUX = "abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/0f8d9b0e-86cc-4454-9772-4ab92eb4db2a/Files/acto/tb_aux.xlsx"


def import_json_payload(arquivo: str): 
    path_json = "/lakehouse/default/Files/acto_gestao_api_payload/" + arquivo

    with open(path_json, "r", encoding="utf-8") as f:
        json_dict = json.load(f)

    json_config = json.dumps(json_dict, ensure_ascii=False)
    return json_config


def extrair_tabela_acto_gestao(arquivo: str, lista_cod_catalogo: list):

    df_etapas = obter_dados_etapa_atual(
        TOKEN, lista_cod_catalogo
    )


    df_solicitacoes = fetch_tabela(import_json_payload(arquivo))

    strings_procuradas = ["logradouro", "Logradouro", "Número", "Agendamento"]

    colunas_encontradas = [
        col for col in df_solicitacoes.columns
        if any(s in col for s in strings_procuradas)
    ]

    df_solicitacoes = df_solicitacoes.drop(columns=colunas_encontradas)

    return df_solicitacoes, df_etapas


def tratar_nome_coluna(df_solicitacoes_etapa):
    # TRATAMENTOS
    # renomear colunas para o padrão
    rename_map = {
        "seqFluxo": "os",
        "Serviço": "servico",
        "Status Fluxo": "status_fluxo",
        "Data Finalização": "data_finalizacao",
        "Data Criação": "data_solicitacao",
        "Solicitante": "solicitante",
        "Bairro": "bairro",
        "Canal": "canal",
        "Tipo de registro": "tipo_registro",
        "etapa": "etapa",
        "executor": "executor",
        "Bairro ocorrência": "bairro_ocorrencia",
        "Bairro interessado": "bairro_interessado"
    }

    df_solicitacoes_etapa = df_solicitacoes_etapa.rename(columns=rename_map)

    # capitalizar o nome do serviço para manter padrão
    df_solicitacoes_etapa['servico'] = df_solicitacoes_etapa['servico'].str.capitalize()

    # transformar datas para o formato correto
    df_solicitacoes_etapa['data_solicitacao'] = pd.to_datetime(df_solicitacoes_etapa['data_solicitacao'], format="ISO8601")
    df_solicitacoes_etapa['data_finalizacao'] = pd.to_datetime(df_solicitacoes_etapa['data_finalizacao'], format="ISO8601")


    # TEMPORÁRIO: voltar para nome da tb_os_acto
    rename_map_bi_asis = {
        "os": "n_da_solicitacao",
        "status_fluxo": "status",
        "data_finalizacao": "data_de_finalizacao",
        "data_solicitacao": "data_de_solicitacao",
        "nome_servico": "nome_do_servico_avaliado",
        "tipo_manifestacao": "tipo_de_manifestacao",
        "bairro": "bairro_consolidado",
        "tipo_registro": "tipo_de_registro",
        "etapa": "etapa_atual",
        "executor": "executor_atual",
    }
    df_solicitacoes_etapa = df_solicitacoes_etapa.rename(columns=rename_map_bi_asis)

    return df_solicitacoes_etapa


def main():

    lista_1 = [
        8255, 8256, 8890, 8910, 8930, 8953, 8958, 8964, 8975, 8976, 8979,
        8984, 9090, 11035, 11044, 11304, 11364, 11434, 11644, 11674,
    ]

    df_solicitacoes1, df_etapas1 = extrair_tabela_acto_gestao(arquivo="payload_sepref1.json", lista_cod_catalogo=lista_1)

    lista_2 = [
        8895, 8898, 8902, 8904, 8912, 8927, 8951, 8956, 8963, 8967, 8979, 
        8980, 11086, 11204, 11205, 11214, 11225, 11534, 11584, 11674,
    ]

    df_solicitacoes2, df_etapas2 = extrair_tabela_acto_gestao(arquivo="payload_sepref2.json", lista_cod_catalogo=lista_2)

    lista_3 = [
        8893, 8896, 8897, 8900, 8960, 8986, 10574, 11085, 11284,
    ]

    df_solicitacoes3, df_etapas3 = extrair_tabela_acto_gestao(arquivo="payload_sepref3.json", lista_cod_catalogo=lista_3)

    df_solicitacoes = pd.concat([df_solicitacoes1, df_solicitacoes2, df_solicitacoes3], axis=0)
    df_etapas = pd.concat([df_etapas1, df_etapas2, df_etapas3], axis=0)

    COLUNAS = [
        "Nº Solicitação", "Serviço", "Data Finalização", "Status Fluxo", 
        "Solicitante", "Data Criação", "Canal", "CPF", "Nome",
        "Bairro ocorrência", "Bairro interessado",
    ]
    for c in COLUNAS:
        df_solicitacoes = aplicar_bfill(df_solicitacoes, c)
        df_solicitacoes = df_solicitacoes.copy()

    
    df_solicitacoes['Bairro ocorrência'] = df_solicitacoes['Bairro ocorrência'].replace("", np.nan)
    df_solicitacoes['bairro_consolidado'] = np.where(
        df_solicitacoes['Bairro ocorrência'].notna(),
        df_solicitacoes['Bairro ocorrência'],
        df_solicitacoes['Bairro interessado']
    )

    df_solicitacoes_etapa = adicionar_etapa_atual_2(df_etapas, df_solicitacoes)

    df_solicitacoes_etapa = tratar_nome_coluna(df_solicitacoes_etapa)

    df_solicitacoes_etapa = harmonizar_nome_bairros(df_solicitacoes_etapa)

    df_solicitacoes_etapa = aplicar_merge_prazo_bairros(df_solicitacoes_etapa)

    df_solicitacoes_etapa = tratar_datas_prazos(df_solicitacoes_etapa)

    df_solicitacoes_etapa = tratar_base_final_solicitacoes(df_solicitacoes_etapa)

    df_solicitacoes_etapa = remover_registros_teste(df_solicitacoes_etapa)

    # deixando igual ao tb_os_acto
    df = spark.sql("SELECT * FROM lh_cidade_inteligente_santos.tb_os_acto LIMIT 10").toPandas()
    df_solicitacoes_etapa = df_solicitacoes_etapa.reindex(columns=df.columns)

    return df_solicitacoes_etapa


df_solicitacoes_etapa = main()

StatementMeta(, 7fe91b3e-73cc-45a2-bcc4-03436fb0c024, 7, Finished, Available, Finished, False)

Status: 200
✅ Linhas em pandas: 1950
Status: 200
✅ Linhas em pandas: 4118
Status: 200
✅ Linhas em pandas: 1708


/tmp/ipykernel_9265/3490408843.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[coluna] = (


In [12]:
sdf_solicitacoes_sepref = spark.createDataFrame(df_solicitacoes_etapa)
(
    sdf_solicitacoes_sepref
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_sepref_servicos")
)

StatementMeta(, e04ee77a-3bc0-4548-89c6-94644a5f4bb3, 20, Finished, Available, Finished)